## PHYS 4811 Project: Using Machine learning to distinguish Leptons from Hadrons using LHCb dataset

#### Author: Fionn Doheny 

#### Student number: 21431185

The first step is to load all the necceccary modules

In [33]:
#importing neccecary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

Then we can load the dataset into pandas

In [34]:
#loading the dataset into pandas
print("Loading dataset...")
training_data = pd.read_csv("training.csv")

#checking if load was successful
print(f"Dataset successfully loaded: Total number of particle tracks: {len(training_data)}")
#print("First 5 rows of dataset:")
#print(training_data.head())

Loading dataset...
Dataset successfully loaded: Total number of particle tracks: 1200000


Since I'm going to be training the algorithm to decipher weather the track which is measured is that of a hadron or a lepton, I'm going to filter the dataset to remove all of the entries where 'Ghost' is seen.

In [35]:
#filters the dataset to remove rows where the label entry is ghost
data_filtered = training_data[training_data['Label'] != 'Ghost'].copy()


Now im going to introduce a new column which is a boolean variable of 1 if the particle is a lepton and 0 if its a hadron

In [36]:
binary_map = {
    'Electron': 1,  # 1 for leptons
    'Muon': 1,      
    'Pion': 0,  # 0 for hadrons
    'Kaon': 0,
    'Proton': 0
}

data_filtered['BinaryClass'] = data_filtered['Label'].map(binary_map)
ignore_cols = {'ID', 'Label', 'Class', 'BinaryClass'} # Ignoring all metadata colums so only numerical measurements remain
features = [col for col in data_filtered.columns if col not in ignore_cols]

#verification
print("\nBinary Class Distribution")
print(data_filtered['BinaryClass'].value_counts())
print(f"Total input features selected: {len(features)}")


Binary Class Distribution
BinaryClass
0    600000
1    400000
Name: count, dtype: int64
Total input features selected: 49


Now im going to use StandardScaler to make sure that the features with massive numerical range (momentum) don't distort the error calculation

In [37]:
# First i extract the raw numpy feature matrices (X) and the target arrays (Y)
X = data_filtered[features].values
y = data_filtered['BinaryClass'].values

# Then i split the data in to training (80%) and validation (20%)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Applying the Scaler
scaler = StandardScaler()

# Fitting and transforming the training features 
# (using fit_transform so that the algorithm learns the statistical data from the training data)
X_train_scaled = scaler.fit_transform(X_train)

# Just transforming the validation data so that there is no data leakage
X_val_scaled = scaler.transform(X_val)

# verification
print(f"training features shape: {X_train_scaled.shape}")
print(f"training features shape: {X_val_scaled.shape}")

training features shape: (800000, 49)
training features shape: (200000, 49)


Now its time to start the actual machine learning!

In [38]:
def predict_particles(X, w, b):
    """
    This will calculate the linear predictions, y_hat on the dataset
    X = a 2D array of the detector features (rows=particles, columns=measurements)
    w = 1D array of the weights
    b = scalar bias term
    """
    y_hat = np.dot(X, w) + b
    return y_hat

Now I need to define a multi Layer perceptron using PyTorch. Im going to use 3 layers starting with an output of 64 features

In [39]:
class LeptonHadronNN(nn.Module):
    def __init__(self, input_dim=49): # because i have 49 elements in my dataset
        super(LeptonHadronNN, self).__init__()

        #Layer 1 going from 49 inputs to 64 neurons
        self.fc1 = nn.Linear(in_features=input_dim, out_features=64) # Linear class type because i started with linear equations y = wx + b
        self.relu = nn.ReLU()

        #Layer 2 going from the 64 neurons to 32
        self.fc2 = nn.Linear(in_features=64, out_features=32)

        #Layer 3 going from 32 to 1 (which is the output score required for binary classification of 1 or 0 (Lepton or Hadron))
        self.fc3 = nn.Linear(in_features=32, out_features=1)

    def forward(self, x):
        # This defines the forward flow so that i start with the 49 features and end up with 1
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

model = LeptonHadronNN(input_dim=49)
print(model)

LeptonHadronNN(
  (fc1): Linear(in_features=49, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
)


Now I have to write the training script for the nural network. Ill use 5 epochs: Forward pass --> compute loss --> zero gradients (so that current calculations are not contaminated with old gradient calcs) -- > backwards pass --> optimse (using adam) to finally reduce the error

In [40]:
# step 1: initiate the loss function and optimizer (since the model has already been defined in the previous cell)
criterion = nn.BCEWithLogitsLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=0.001)

# step 2: convert scaled numpy arrays into PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1) # so the shape is (N, 1)

# Training loop over the 5 epochs:
epochs = 5
for epoch in range(epochs):
    # A: clear old gradients
    optimiser.zero_grad()

    #B: Forward pass to make the predictions
    outputs = model(X_train_tensor)

    #C: Compute loss by comparing with true data
    loss = criterion(outputs, y_train_tensor)

    #D: Backwards pass to compute gradients using autograd
    loss.backward()

    #E: Optimiser step (updating model weights)
    optimiser.step()

    # confirmation
    print(f"Echoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Echoch [1/5], Loss: 0.6950
Echoch [2/5], Loss: 0.6905
Echoch [3/5], Loss: 0.6861
Echoch [4/5], Loss: 0.6819
Echoch [5/5], Loss: 0.6777


Now its time to test the algorithm on the validation data that we split from the original dataset right near the begining! Note that im using the 'torch.no_grad()' here because I have already finalised the weights and so i dont want to continue calculating gradients and changing weights with this data.

In [41]:
# first i must make sure that the model is in evaluation mode
model.eval()

# then i have to convert the validation NumPy arrays into PyTorch Tensors
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

# then i can evaluate the data without computing the gradients
with torch.no_grad():
    # getting the raw output scores (logits) from the model
    val_logits = model(X_val_tensor)

    # passing the logits through the sigmoid to get continuous probabilities between 0 and 1
    val_probs = torch.sigmoid(val_logits)

    # converting the probabilities to binary predictions 
    val_preds = (val_probs >= 0.5).float()

    # finally i calculate the accuracy
    correct = (val_preds == y_val_tensor).sum().item()
    accuracy = correct / y_val_tensor.shape[0]

print(f"Validation Accuracy: {accuracy * 100:.2f}%")

Validation Accuracy: 66.15%


The first time i ran this it yeilded a validation accuracy of 59.92% which is pretty low. This is likely because of the fact that after only 5 epochs (in combination with a learning rate of only 0.001) the algorithm didnt have enough time to actually learn anything. I'm going to increase the number of epochs in the cell below to see if it makes a difference

In [42]:
epochs = 50  # Increased from 5 to 50
optimiser = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(epochs):
    optimiser.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimiser.step()
    
    # print every 10 epochs so it doesn't spam the screen
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [1/50], Loss: 0.6735
Epoch [10/50], Loss: 0.6314
Epoch [20/50], Loss: 0.5748
Epoch [30/50], Loss: 0.5136
Epoch [40/50], Loss: 0.4673
Epoch [50/50], Loss: 0.4333


Again, I'm going to copy the validation cell to see if there is a difference in the accuracy now 

In [43]:
model.eval()

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

with torch.no_grad():
    val_logits = model(X_val_tensor)
    val_probs = torch.sigmoid(val_logits)
    val_preds = (val_probs >= 0.5).float()
    correct = (val_preds == y_val_tensor).sum().item()
    accuracy = correct / y_val_tensor.shape[0]

print(f"Validation Accuracy: {accuracy * 100:.2f}%")

Validation Accuracy: 78.33%


Awesome thats a good increase from 59.92% to 78.88%. More training time is bound to do this.

An additional way that i can try to increase the accuracy is to increase the number of layers in my perceptron from 3 to say 5. To do this Ill start with 256 features and go down from there to 128 then 64 then 32 then 1. These numbers are used because they are powers of 2 which computers find easier to work with.

In [44]:
#redefining the model
class LeptonHadronNN(nn.Module):
    def __init__(self, input_dim=49): # because i have 49 elements in my dataset
        super(LeptonHadronNN, self).__init__()

        #Layer 1 going from 49 inputs to 256 neurons
        self.fc1 = nn.Linear(in_features=input_dim, out_features=256) # Linear class type because i started with linear equations y = wx + b
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(in_features=256, out_features=128)
        self.fc3 = nn.Linear(in_features=128, out_features=64)
        self.fc4 = nn.Linear(in_features=64, out_features=32)
        self.fc5 = nn.Linear(in_features=32, out_features=1)

    def forward(self, x):
        # Re-defining the forward flow to add in the extra layers with more neurons
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.relu(x)
        x = self.fc4(x)
        x = self.relu(x)
        x = self.fc5(x)
        return x

model = LeptonHadronNN(input_dim=49)

In [45]:
#retraining it
epochs = 50  
optimiser = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(epochs):
    optimiser.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimiser.step()
    
    # print every 10 epochs so it doesn't spam the screen
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [1/50], Loss: 0.6868
Epoch [10/50], Loss: 0.6237
Epoch [20/50], Loss: 0.4971
Epoch [30/50], Loss: 0.3931
Epoch [40/50], Loss: 0.3393
Epoch [50/50], Loss: 0.2906


In [46]:
#re-validating it
model.eval()

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

with torch.no_grad():
    val_logits = model(X_val_tensor)
    val_probs = torch.sigmoid(val_logits)
    val_preds = (val_probs >= 0.5).float()
    correct = (val_preds == y_val_tensor).sum().item()
    accuracy = correct / y_val_tensor.shape[0]

print(f"Validation Accuracy: {accuracy * 100:.2f}%")


Validation Accuracy: 88.49%


Even better! Its gone from 78.88% to 87.12% a pretty nice increase.

For a while i had been running this new model but had forgetten to redefine the optimiser which had meant that the weights were remaining fixed. This meant that whenever i ran this new model and the ones after it, my accuracy kept on decreasing. now however, after noticing it, my accuracy is increasing (as it should be).
Because of this i wanted to test whether adding a dropout layer would make my accuracy better than the previously achieved 78.88%, however as you will see, it didnt.
This cell is pretty much redundant now as i wouldnt have coded it to check if hadnt forgetten to redefine the optimiser when training.

In [47]:
#re-redfining the model

class LeptonHadronNN(nn.Module):
    def __init__(self, input_dim=49): # because i have 49 elements in my dataset
        super(LeptonHadronNN, self).__init__()

        self.fc1 = nn.Linear(in_features=input_dim, out_features=64) 
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1) # dropout layer to randomly turn off 30% of the data
        self.fc2 = nn.Linear(in_features=64, out_features=32)
        self.fc3 = nn.Linear(in_features=32, out_features=1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x) # Apply dropout after the activation
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

# Instantiate the model
model = LeptonHadronNN(input_dim=49)

In [ ]:
#re-retraining it
epochs = 50  
optimiser = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(epochs):
    optimiser.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimiser.step()
    
    # print every 10 epochs so it doesn't spam the screen
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [1/50], Loss: 0.6896
Epoch [10/50], Loss: 0.6559
Epoch [20/50], Loss: 0.6098


In [ ]:
#re-revalidating it
model.eval()

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

with torch.no_grad():
    val_logits = model(X_val_tensor)
    val_probs = torch.sigmoid(val_logits)
    val_preds = (val_probs >= 0.5).float()
    correct = (val_preds == y_val_tensor).sum().item()
    accuracy = correct / y_val_tensor.shape[0]

print(f"Validation Accuracy: {accuracy * 100:.2f}%")


Validation Accuracy: 77.87%


This relative to the 78.88% which was originally achieved with 50 epochs and 3 levels makes sense, dropping a small amount of the data is bound to make th ealgorithm slightly less acurate. Thus this is no use to me.

Going back to the 5 layer perceptron, I'm going to introduce batch normalisation.
Batch normalisation makes learning easier for the algorithm because it scales the data between layers towards zero-mean and unit-variance 

In [ ]:
class LeptonHadronNN(nn.Module):
    def __init__(self, input_dim=49):
        super(LeptonHadronNN, self).__init__()
        
        # Layer 1
        self.fc1 = nn.Linear(input_dim, 256)
        self.bn1 = nn.BatchNorm1d(256) # normalizes the 256 outputs
        self.relu = nn.ReLU()

        # Layer 2
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128) # normalizes the 128 outputs
        
        # Layer 3
        self.fc3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)  # normalizes the 64 outputs
        
        # Layer 4
        self.fc4 = nn.Linear(64, 32)
        self.bn4 = nn.BatchNorm1d(32)  # normalizes the 32 outputs
        
        # Layer 5 (Output)
        self.fc5 = nn.Linear(32, 1) 

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu(x)
        
        x = self.fc3(x)
        x = self.bn3(x)
        x = self.relu(x)
        
        x = self.fc4(x)
        x = self.bn4(x)
        x = self.relu(x)
        
        x = self.fc5(x)
        return x

model = LeptonHadronNN(input_dim=49)

optimiser = torch.optim.Adam(model.parameters(), lr=0.001) #defining the optimser here so i dont forget to add it in the training cell lol

In [ ]:
epochs = 50  

for epoch in range(epochs):
    optimiser.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimiser.step()
    
    # print every 10 epochs so it doesn't spam the screen
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [1/50], Loss: 0.7123
Epoch [10/50], Loss: 0.4398
Epoch [20/50], Loss: 0.3636
Epoch [30/50], Loss: 0.3084
Epoch [40/50], Loss: 0.2650
Epoch [50/50], Loss: 0.2339


In [ ]:
# rvalidation!
model.eval()

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

with torch.no_grad():
    val_logits = model(X_val_tensor)
    val_probs = torch.sigmoid(val_logits)
    val_preds = (val_probs >= 0.5).float()
    correct = (val_preds == y_val_tensor).sum().item()
    accuracy = correct / y_val_tensor.shape[0]

print(f"Validation Accuracy: {accuracy * 100:.2f}%")


Validation Accuracy: 89.68%


The final thing im going to implement to try to improve this accuracy even more is early stopping.
Even though i dont think, off instinct, that 50 epochs is overtraining, I'm going to put a max of 100 and see if the optimal number is greater than 50.
Early stopping will test and check the validation gradient to see if its getting better with every change of weights. If it is then itll continue, if its not, then it will fix the wieghts at the level they were at before the decrease accuracy occured.

I'm going to re-load the algorithm without the dropout layer

In [ ]:
class LeptonHadronNN(nn.Module):
    def __init__(self, input_dim=49):
        super(LeptonHadronNN, self).__init__()
        
        # Layer 1
        self.fc1 = nn.Linear(input_dim, 256)
        self.bn1 = nn.BatchNorm1d(256) # normalizes the 256 outputs
        self.relu = nn.ReLU()

        # Layer 2
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128) # normalizes the 128 outputs
        
        # Layer 3
        self.fc3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)  # normalizes the 64 outputs
        
        # Layer 4
        self.fc4 = nn.Linear(64, 32)
        self.bn4 = nn.BatchNorm1d(32)  # normalizes the 32 outputs
        
        # Layer 5 (Output)
        self.fc5 = nn.Linear(32, 1) 

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu(x)
        
        x = self.fc3(x)
        x = self.bn3(x)
        x = self.relu(x)
        
        x = self.fc4(x)
        x = self.bn4(x)
        x = self.relu(x)
        
        x = self.fc5(x)
        return x

model = LeptonHadronNN(input_dim=49)

optimiser = torch.optim.Adam(model.parameters(), lr=0.001) #defining the optimser here so i dont forget to add it in the training cell lol

In [ ]:
import copy #this is needed here to make sure the algorithm can go back if need be

epochs = 1000 # setting a max limit
patience = 10 # keeps track of how many consecutive epochs have failed at improving loss
best_val_loss = float('inf') # initially loss should be very high so that it doesnt stop at the begining
patience_counter = 0 # initially 0
best_model_weights = None

for epoch in range(epochs):

    #training phase
    model.train()
    optimiser.zero_grad() #clears the old gradients from the prev step
    train_outputs = model(X_train_tensor) # passes data forward
    train_loss = criterion(train_outputs, y_train_tensor) # calculates how wrong
    train_loss.backward() # calculates new gradients
    optimiser.step() # updates the wieghts

    #validation phase
    model.eval()
    with torch.no_grad(): # turinging off gradient tracking to increase speed
        val_outputs = model(X_val_tensor) # passing unseen data through network
        val_loss = criterion(val_outputs, y_val_tensor) #calculates loss based on validation data

    #early stopping check
    # check if validation loss from this epoch is lower than the current record
    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item() #update the record low
        patience_counter = 0 # reset the patience counter because the model has improved
        best_model_weights = copy.deepcopy(model.state_dict()) # takes lil snapshot of the exact weights at this point and saves them
    else:
        patience_counter += 1 # if model doesnt improve, just increase counter

    #print every 20 
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss.item():.3f} | Val Loss: {val_loss.item():.3f} | Patience: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print(f"\nEarly stopping triggered at epoch {epoch+1}! Restoring best weights.")
        # reload the snapshot saved
        model.load_state_dict(best_model_weights)
        break # exit the training loop immidiately

Epoch [1/1000] | Train Loss: 0.759 | Val Loss: 0.712 | Patience: 0/10
Epoch [20/1000] | Train Loss: 0.432 | Val Loss: 0.514 | Patience: 0/10
Epoch [40/1000] | Train Loss: 0.324 | Val Loss: 0.366 | Patience: 0/10
Epoch [60/1000] | Train Loss: 0.259 | Val Loss: 0.301 | Patience: 1/10
Epoch [80/1000] | Train Loss: 0.220 | Val Loss: 0.238 | Patience: 1/10
Epoch [100/1000] | Train Loss: 0.192 | Val Loss: 0.195 | Patience: 1/10
Epoch [120/1000] | Train Loss: 0.174 | Val Loss: 0.177 | Patience: 2/10
Epoch [140/1000] | Train Loss: 0.165 | Val Loss: 0.168 | Patience: 1/10
Epoch [160/1000] | Train Loss: 0.153 | Val Loss: 0.155 | Patience: 1/10
Epoch [180/1000] | Train Loss: 0.145 | Val Loss: 0.149 | Patience: 1/10
Epoch [200/1000] | Train Loss: 0.139 | Val Loss: 0.144 | Patience: 1/10
Epoch [220/1000] | Train Loss: 0.134 | Val Loss: 0.137 | Patience: 0/10
Epoch [240/1000] | Train Loss: 0.129 | Val Loss: 0.133 | Patience: 1/10
Epoch [260/1000] | Train Loss: 0.126 | Val Loss: 0.131 | Patience: 1/1

In [ ]:
# rvalidation!
model.eval()

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

with torch.no_grad():
    val_logits = model(X_val_tensor)
    val_probs = torch.sigmoid(val_logits)
    val_preds = (val_probs >= 0.5).float()
    correct = (val_preds == y_val_tensor).sum().item()
    accuracy = correct / y_val_tensor.shape[0]

print(f"Validation Accuracy: {accuracy * 100:.2f}%")


Validation Accuracy: 95.14%


Ok! that took a LOT longer than i expected but in the end I have a very high accuracy figure of 95.14% this is pretty good.